# SP&L feeder generation and structural checks

Run the existing SP&L infrastructure generator in a temporary directory, load its
six core tables, and check how asset relationships become a network graph.
The repository's checked-in datasets are read for fingerprinting and are never overwritten.

The generated assets are fictional and use the existing Phoenix-referenced,
straight-line coordinate model. This notebook does not import streets or generate
a Golden, Colorado network. The final section explains where that extension belongs.

This example generates **substations, feeders, transformers, customers, network
nodes, and network edges**. It omits time series, DER, V2 conversion, and electrical
studies. Because the full generator shares one random stream across all stages,
the network attributes here differ from the published full-pipeline dataset.
Reproducibility is checked for this notebook's explicit stage order.

## Setup

Use Python 3.12 or newer. Clone the repository and open this notebook from the
repository root or its `notebooks/` directory. In your notebook environment:

```bash
python -m pip install pandas networkx pyarrow jupyterlab ipykernel
jupyter lab
```

The original read-only companion is
[`feeder_walkthrough.py`](../examples/feeder-generation/feeder_walkthrough.py).
It checks the published dataset through the combined V1/V2 loader. This notebook
instead loads newly generated V1 tables in isolation.

In [1]:
import hashlib
import importlib.util
import json
from pathlib import Path
import platform
import subprocess
import sys
import tempfile

import networkx as nx
import pandas as pd

REPO = next(
    (p for p in (Path.cwd(), *Path.cwd().parents)
     if (p / "demo_data/generate_demo_data.py").is_file()),
    None,
)
if REPO is None:
    raise RuntimeError("Open this notebook from a Dynamic-Network-Model checkout.")

GENERATOR = REPO / "demo_data/generate_demo_data.py"
CORE = ("substations", "feeders", "transformers", "customers", "network_nodes", "network_edges")
SEED = 42

def sha256(path):
    with path.open("rb") as stream:
        return hashlib.file_digest(stream, "sha256").hexdigest()

source_hashes = {name: sha256(REPO / "demo_data" / f"{name}.csv") for name in CORE}
generator_hash = sha256(GENERATOR)
revision = subprocess.run(
    ["git", "-C", str(REPO), "rev-parse", "HEAD"],
    check=True, capture_output=True, text=True,
).stdout.strip()
print(json.dumps({"source_revision": revision, "generator_sha256": generator_hash,
                  "python": platform.python_version(), "pandas": pd.__version__,
                  "networkx": nx.__version__, "seed": SEED}, indent=2))

{
  "source_revision": "580909edc9cf7233195cdcd52d1b65f40f02eb48",
  "generator_sha256": "852667de73e28194f8fbaa8fb671130d151b6fcfd240a083c84250542c9de2ff",
  "python": "3.12.13",
  "pandas": "2.3.2",
  "networkx": "3.5",
  "seed": 42
}


## Generate an isolated infrastructure example

A separate Python process runs the existing generator functions. The output
directory is explicitly redirected to a fresh temporary folder before any
generation function runs. The order matches the infrastructure portion of
`generate_demo_data.main()`, including its correction of feeder customer counts.

All six generated files remain available through `CASE_DIR` while the kernel is
running. Copy that directory to a chosen destination if you want to retain them.

In [2]:
GENERATION_CODE = r"""
from collections import Counter
import csv
import importlib.util
from pathlib import Path
import sys

source, destination, seed = Path(sys.argv[1]), Path(sys.argv[2]), int(sys.argv[3])
spec = importlib.util.spec_from_file_location("spl_generator", source)
generator = importlib.util.module_from_spec(spec)
spec.loader.exec_module(generator)
generator.OUTPUT_DIR = str(destination)
generator.random.seed(seed)

substations = generator.generate_substations()
feeders = generator.generate_feeders(substations)
transformers = generator.generate_transformers(feeders)
customers = generator.generate_customers(transformers, feeders)

with (destination / "customers.csv").open(newline="") as stream:
    counts = Counter(row["feeder_id"] for row in csv.DictReader(stream))
with (destination / "feeders.csv").open(newline="") as stream:
    headers = next(csv.reader(stream))
feeder_id_index = headers.index("feeder_id")
customer_count_index = headers.index("num_customers")
for feeder in feeders:
    feeder[customer_count_index] = counts[feeder[feeder_id_index]]
generator.write_csv("feeders.csv", headers, feeders)

generator.generate_network_nodes_and_edges(substations, feeders, transformers)
"""

def generate_case(destination):
    destination.mkdir(parents=True, exist_ok=False)
    result = subprocess.run(
        [sys.executable, "-B", "-c", GENERATION_CODE,
         str(GENERATOR), str(destination), str(SEED)],
        check=True, capture_output=True, text=True,
    )
    return result.stdout

case_workspace = tempfile.TemporaryDirectory(prefix="spl-feeder-generation-")
CASE_DIR = Path(case_workspace.name) / "demo_data"
print(generate_case(CASE_DIR))

Generating substations...
  wrote         23 rows -> substations.csv
Generating feeders...
  wrote        104 rows -> feeders.csv
Generating transformers...
  wrote     36,668 rows -> transformers.csv
Generating customers...
  wrote    237,713 rows -> customers.csv
  wrote        104 rows -> feeders.csv
Generating network nodes...
  adding tie switches...
  wrote     74,517 rows -> network_nodes.csv
Generating network edges...
  wrote     74,529 rows -> network_edges.csv



## Read the generated tables through the existing loader

The normal loader combines selected V1 and V2 datasets. For this example, create
a separate loader module whose V1 directory is the scratch case and explicitly
disable V2. This prevents the new network from being mixed with the repository's
existing health and time-series records.

The hierarchy is `substation_id -> feeder_id -> transformer_id -> customer_id`.
Graph edges reference `network_nodes.node_id`; each transformer node uses the
same ID as its asset record. Customers have coordinates and transformer
assignments, but individual service-drop edges are outside this graph.

In [3]:
spec = importlib.util.spec_from_file_location(
    "scratch_spl_loader", REPO / "demo_data/load_demo_data.py"
)
scratch_loader = importlib.util.module_from_spec(spec)
spec.loader.exec_module(scratch_loader)
scratch_loader.V1_DIR = str(CASE_DIR)
scratch_loader.DATA_DIR = str(CASE_DIR)
scratch_loader.USE_V2 = False
data = scratch_loader.load_all(list(CORE))

pd.DataFrame([
    {"table": name, "rows": len(frame), "primary_key": frame.index.name}
    for name, frame in data.items()
])

,table,rows,primary_key
0,substations,23,substation_id
1,feeders,104,feeder_id
2,transformers,36668,transformer_id
3,customers,237713,customer_id
4,network_nodes,74517,node_id
5,network_edges,74529,edge_id


## Verify the asset hierarchy and structural graph

These checks establish referential integrity and tree structure. A structural
tree includes the open tail segment and may include equipment marked failed.
Energized connectivity requires switch states, equipment availability, and phase
continuity. Electrical feasibility requires a separate power-flow study.

Units: coordinates are latitude/longitude degrees; edge lengths are miles and
feet; resistance/reactance are ohms per mile; transformer ratings are kVA;
customer contracted demand is kW. Contracted demand is not coincident loading.

In [4]:
subs, feeders, transformers, customers, nodes, edges = (data[name] for name in CORE)
assert all(frame.index.is_unique for frame in data.values())
assert feeders.substation_id.isin(subs.index).all()
assert transformers.feeder_id.isin(feeders.index).all()
assert transformers.substation_id.eq(transformers.feeder_id.map(feeders.substation_id)).all()
assert customers.transformer_id.isin(transformers.index).all()
assert customers.feeder_id.eq(customers.transformer_id.map(transformers.feeder_id)).all()
assert customers.substation_id.eq(customers.transformer_id.map(transformers.substation_id)).all()
counts = customers.groupby("feeder_id").size().reindex(feeders.index, fill_value=0)
assert feeders.num_customers.eq(counts).all()
assert edges.from_node_id.isin(nodes.index).all()
assert edges.to_node_id.isin(nodes.index).all()
assert edges.feeder_id.isin(feeders.index).all()
assert transformers.index.isin(nodes.index).all()
pd.testing.assert_frame_equal(
    transformers[["latitude", "longitude"]],
    nodes.loc[transformers.index, ["latitude", "longitude"]],
    check_names=False,
)

feeder_checks = []
for feeder_id, feeder in feeders.iterrows():
    selected = edges[edges.feeder_id.eq(feeder_id) & edges.edge_type.ne("tie")]
    graph = nx.from_pandas_edgelist(
        selected, "from_node_id", "to_node_id", create_using=nx.MultiGraph()
    )
    assert feeder.substation_id in graph, feeder_id
    assert nx.is_tree(graph), feeder_id
    members = transformers[transformers.feeder_id.eq(feeder_id)]
    assert members.index.isin(list(graph.nodes)).all(), feeder_id
    feeder_checks.append({
        "feeder_id": feeder_id, "substation_id": feeder.substation_id,
        "transformers": len(members), "customers": int(counts.loc[feeder_id]),
        "nodes": graph.number_of_nodes(), "edges": graph.number_of_edges(),
        "structural_tree": True,
    })

print(f"Hierarchy and endpoint checks passed; {len(feeder_checks)} structural feeder trees.")
pd.DataFrame(feeder_checks).head()

Hierarchy and endpoint checks passed; 104 structural feeder trees.


,feeder_id,substation_id,transformers,customers,nodes,edges,structural_tree
0,FDR-0001,SUB-001,364,2372,740,739,True
1,FDR-0002,SUB-001,157,1028,326,325,True
2,FDR-0003,SUB-001,286,1897,584,583,True
3,FDR-0004,SUB-002,409,2586,830,829,True
4,FDR-0005,SUB-002,498,3190,1008,1007,True


## Check repeatability and preserve the source dataset

Generate the same six tables again in another temporary directory and compare
their SHA-256 hashes. This checks repeatability for the current generator,
Python version, seed, and stage order. It does not assert equivalence to the
full generator, whose intervening stages consume additional random draws.

In [5]:
generated_hashes = {name: sha256(CASE_DIR / f"{name}.csv") for name in CORE}
with tempfile.TemporaryDirectory(prefix="spl-feeder-repeat-") as directory:
    repeat_dir = Path(directory) / "demo_data"
    generate_case(repeat_dir)
    repeat_hashes = {name: sha256(repeat_dir / f"{name}.csv") for name in CORE}
assert generated_hashes == repeat_hashes
assert source_hashes == {
    name: sha256(REPO / "demo_data" / f"{name}.csv") for name in CORE
}
assert generator_hash == sha256(GENERATOR)
print("Repeat generation matched all six files; checked-in infrastructure and generator unchanged.")
print(json.dumps({"seed": SEED, "generator_sha256": generator_hash,
                  "generated_sha256": generated_hashes}, indent=2))

Repeat generation matched all six files; checked-in infrastructure and generator unchanged.
{
  "seed": 42,
  "generator_sha256": "852667de73e28194f8fbaa8fb671130d151b6fcfd240a083c84250542c9de2ff",
  "generated_sha256": {
    "substations": "8fe34a35393fa603f0cfe60aee79e60b19640f65cec272afff628e3087a7f7fd",
    "feeders": "a005a3510552175429e81b6b8bf77a6e5e24b611f9bcd2e117af7792af32958e",
    "transformers": "0f56cd5b0b9050a475f23ccad7d7fee48af35f056829f2fdceba6ced66251755",
    "customers": "3acf6a5ca9654a42f5c547dca495df4318c68653a95657b69c638c7b65b23167",
    "network_nodes": "b1f2e71f4ebc68bb36e0455c16307261e914bd8a8bb5f01502a365077451055d",
    "network_edges": "def6e560c77c624d2f957efaf292324d53abe20860dfd623ca4905d5fe165f81"
  }
}


## Where a Golden street-based extension fits

Keep the explicit asset IDs and node/edge relationships. Replace the current
coordinate placement **and** graph-construction rules with a street-constrained
workflow. Moving the existing points to Golden does not make the lines follow
Golden's streets.

1. Define the study boundary, assumed source locations, and customer/demand
   inputs. Record which inputs are observed and which are synthetic.
2. Acquire a street graph and project it to a suitable local metric coordinate
   system. Street corridors provide possible routes; driving directions and
   physical crossings do not determine electrical connections.
3. Assign customers to transformers, select feeder branches along permitted
   corridors, and retain a connected radial operating topology. Store each
   route's full geometry in a spatial layer keyed by `edge_id`; derive lengths
   from that geometry rather than endpoint distance.
4. Export compatible asset/node/edge tables and validate references, phase
   continuity, switch states, and equipment availability. Size equipment against
   explicit demand and DER assumptions before interpreting power-flow results.

[OSMnx](https://osmnx.readthedocs.io/en/stable/getting-started.html) supplies street
graphs, routing, and retained road geometry. See
[GeoPandas projections](https://geopandas.org/en/stable/docs/user_guide/projections.html)
for coordinate handling.

The current SP&L OpenDSS exporter needs additional state handling: non-tie edges
marked open and transformers marked failed are still included as enabled
equipment. The reference base case also has voltage violations. Those limits
are separate from the structural checks demonstrated here. See
[`DATA-QUALITY.md`](../DATA-QUALITY.md) for the electrical study boundary.

For byte-for-byte reproduction of the published dataset, retain the reference
revision, environment, and checked-in artifacts. Running all V1 generator stages
also does not rebuild the V2 conversions or electrical results.